# Día 3 — Notebook Local: Blob Storage + PostgreSQL + PostGIS
### Curso: Introducción a Herramientas de Cómputo en la Nube · Azure

---

Este notebook se ejecuta **en tu computadora**. Cubre:

| Sección | Servicio | Actividad |
|---------|----------|-----------|
| 0 | — | Instalación de librerías |
| 1 | — | Configuración de credenciales |
| 2 | Blob Storage | Conectar, listar, descargar CSV |
| 3 | PostgreSQL | Conectar, crear tabla, insertar datos |
| 4 | PostGIS | Activar extensión, insertar puntos, consultas geoespaciales |
| 5 | — | Pipeline integrado Blob → PostgreSQL |

> El notebook de **Azure Databricks** es un archivo separado: `dia3_databricks.ipynb`

---
## Sección 0 — Instalación de librerías
Ejecuta esta celda una sola vez. Si ya están instaladas puedes saltarla.

In [1]:
%pip install adlfs fsspec psycopg2-binary pandas geopandas shapely adlfs --quiet
print('Listo.')

Note: you may need to restart the kernel to use updated packages.
Listo.


---
## Sección 1 — Credenciales

Rellena estas variables con tus datos de Azure antes de continuar.

**Dónde encontrarlos en el portal:**
- Blob connection string: `Storage Account > Access keys > Connection string`
- PostgreSQL host: `Azure Database for PostgreSQL > Overview > Server name`
- PostgreSQL password: la que definiste al crear el servidor

In [2]:
# ── BLOB STORAGE ──────────────────────────────────────────────────────────────
BLOB_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=satallerazuredia2;AccountKey=yxKStK+i7NZCkh5oXiSLuTD9e9yQik35G7uzZHV8ohFoz35AfYvdwNDEXq6Sws3K4hBdtF3kOIY0+ASt85Bm2g==;EndpointSuffix=core.windows.net"
BLOB_CONTAINER         = "crudos"   # nombre del contenedor del Día 1
BLOB_ARCHIVO_CSV       = "datos.csv"       # archivo CSV que subiste al contenedor

# ── POSTGRESQL + POSTGIS ──────────────────────────────────────────────────────
PG_HOST     = "db-taller-dia3.postgres.database.azure.com"
PG_PORT     = 5432
PG_DATABASE = "postgres"        # nombre de la base de datos
PG_USER     = "azure_taller_lizzie"      # usuario administrador que definiste
PG_PASSWORD = "zy2I8$b7fZCc"
PG_SSLMODE  = "require"         # Azure PostgreSQL requiere SSL

print('Credenciales configuradas.')
print(f'  Blob container : {BLOB_CONTAINER}')
print(f'  PG host        : {PG_HOST}')

Credenciales configuradas.
  Blob container : crudos
  PG host        : db-taller-dia3.postgres.database.azure.com


---
## Sección 2 — Azure Blob Storage con Python

Vamos a:
1. Conectarnos al Storage Account
2. Listar los archivos del contenedor
3. Descargar el CSV y cargarlo como DataFrame

In [3]:
import adlfs
import pandas as pd

# adlfs implementa la interfaz fsspec para Azure Blob Storage.
# Extraemos account_name y account_key desde la connection string.
def _parse_conn_str(cs):
    parts = dict(p.split('=', 1) for p in cs.split(';') if '=' in p)
    return parts.get('AccountName'), parts.get('AccountKey')

ACCOUNT_NAME, ACCOUNT_KEY = _parse_conn_str(BLOB_CONNECTION_STRING)

# Instanciar el filesystem
fs = adlfs.AzureBlobFileSystem(
    account_name=ACCOUNT_NAME,
    account_key=ACCOUNT_KEY
)

print(f'Filesystem fsspec listo.')
print(f'  Account : {ACCOUNT_NAME}')
print(f'  Backend : {type(fs).__name__}')

Filesystem fsspec listo.
  Account : satallerazuredia2
  Backend : AzureBlobFileSystem


In [4]:
# Listar archivos del contenedor con fsspec
archivos = fs.ls(BLOB_CONTAINER)
print(f'Archivos en el contenedor ({len(archivos)} total):')
for a in archivos:
    info = fs.info(a)
    print(f'  {info["name"].split("/", 1)[-1]}  ({info["size"] / 1024:.1f} KB)')

Archivos en el contenedor (2 total):
  datos.csv  (3.3 KB)
  subnets.csv  (5.3 KB)


In [5]:
# Leer el CSV directamente con fsspec — sin descargar, sin BytesIO
# pd.read_csv acepta cualquier objeto compatible con fsspec
ruta = f'{BLOB_CONTAINER}/{BLOB_ARCHIVO_CSV}'

with fs.open(ruta, 'rb') as f:
    df = pd.read_csv(f)

print(f'Archivo leido: {BLOB_ARCHIVO_CSV}')
print(f'  Filas    : {len(df)}')
print(f'  Columnas : {list(df.columns)}')
df.head()

Archivo leido: datos.csv
  Filas    : 80
  Columnas : ['fecha', 'nombre', 'valor', 'categoria']


,fecha,nombre,valor,categoria
0,2024-01-02,Sofia Herrera,1333.36,Tecnologia
1,2024-01-04,Isabella Romero,4031.04,Tecnologia
2,2024-01-04,Carlos Ruiz,3542.13,Ventas
3,2024-01-06,Camila Flores,3606.40,Finanzas
4,2024-01-08,Carlos Ruiz,4632.57,Marketing


In [6]:
print('Tipos de datos por columna:')
print(df.dtypes)
print()
print('Valores nulos por columna:')
print(df.isnull().sum())

Tipos de datos por columna:
fecha            str
nombre           str
valor        float64
categoria        str
dtype: object

Valores nulos por columna:
fecha        0
nombre       0
valor        0
categoria    0
dtype: int64


---
## Sección 3 — Azure Database for PostgreSQL

**Pasos que hiciste en el portal antes de llegar aquí:**
1. Crear Flexible Server (tier Burstable B1ms)
2. En `Networking`: habilitar acceso público y agregar tu IP en el firewall
3. En `Server parameters`: asegurarse de que `require_secure_transport = ON`

Ahora nos conectamos con Python.

In [7]:
import psycopg2

conn = psycopg2.connect(
    host     = PG_HOST,
    port     = PG_PORT,
    dbname   = PG_DATABASE,
    user     = PG_USER,
    password = PG_PASSWORD,
    sslmode  = PG_SSLMODE
)
conn.autocommit = False
cur = conn.cursor()

cur.execute('SELECT version();')
version = cur.fetchone()[0]
print(f'Conectado a PostgreSQL:')
print(f'  {version[:60]}...')

Conectado a PostgreSQL:
  PostgreSQL 17.7 on x86_64-pc-linux-gnu, compiled by gcc (GCC...


In [8]:
# Crear tabla para los datos del CSV
# Ajusta las columnas si tu CSV tiene nombres diferentes

cur.execute("""
    CREATE TABLE IF NOT EXISTS registros (
        id        SERIAL PRIMARY KEY,
        fecha     TEXT,
        nombre    TEXT,
        valor     FLOAT,
        categoria TEXT,
        creado_en TIMESTAMP DEFAULT NOW()
    )
""")
conn.commit()
print('Tabla "registros" creada (o ya existia).')

Tabla "registros" creada (o ya existia).


In [9]:
# Insertar datos del DataFrame en PostgreSQL
# Ajusta los indices de columna (.iloc[0], .iloc[1]...) segun tu CSV

insert_sql = """
    INSERT INTO registros (fecha, nombre, valor, categoria)
    VALUES (%s, %s, %s, %s)
"""

filas = []
for _, row in df.iterrows():
    filas.append((
        str(row.iloc[0]),
        str(row.iloc[1]),
        float(row.iloc[2]) if pd.notna(row.iloc[2]) else None,
        str(row.iloc[3]) if len(row) > 3 else None
    ))

cur.executemany(insert_sql, filas)
conn.commit()
print(f'{len(filas)} filas insertadas en la tabla "registros".')

80 filas insertadas en la tabla "registros".


In [10]:
# Consultar los datos insertados
cur.execute('SELECT * FROM registros ORDER BY id DESC LIMIT 10')
cols = [desc[0] for desc in cur.description]
rows = cur.fetchall()

df_pg = pd.DataFrame(rows, columns=cols)
print(f'Ultimos 10 registros en PostgreSQL:')
df_pg

Ultimos 10 registros en PostgreSQL:


,id,fecha,nombre,valor,categoria,creado_en
0,80,2024-12-25,Daniela Soto,3420.90,Ventas,2026-03-04 15:29:14.408919
1,79,2024-12-12,Luis Navarro,4461.98,Ventas,2026-03-04 15:29:14.408919
2,78,2024-12-10,Camila Flores,4920.98,Ventas,2026-03-04 15:29:14.408919
3,77,2024-12-08,Luis Navarro,1577.78,Operaciones,2026-03-04 15:29:14.408919
4,76,2024-12-04,Luis Navarro,2438.49,Marketing,2026-03-04 15:29:14.408919
5,75,2024-11-25,Diego Morales,722.93,Marketing,2026-03-04 15:29:14.408919
6,74,2024-11-23,Carlos Ruiz,134.80,Operaciones,2026-03-04 15:29:14.408919
7,73,2024-11-07,Jorge Mendez,772.67,Marketing,2026-03-04 15:29:14.408919
8,72,2024-11-07,Camila Flores,863.98,Marketing,2026-03-04 15:29:14.408919
9,71,2024-11-02,Pedro Castillo,2448.14,Ventas,2026-03-04 15:29:14.408919


In [11]:
# Estadisticas directamente desde SQL
cur.execute("""
    SELECT
        COUNT(*)          AS total,
        ROUND(AVG(valor)::numeric, 2) AS promedio,
        MAX(valor)        AS maximo,
        MIN(valor)        AS minimo
    FROM registros
""")
r = cur.fetchone()
print('Estadisticas desde PostgreSQL:')
print(f'  Total registros : {r[0]}')
print(f'  Promedio valor  : {r[1]}')
print(f'  Maximo          : {r[2]}')
print(f'  Minimo          : {r[3]}')

Estadisticas desde PostgreSQL:
  Total registros : 80
  Promedio valor  : 2378.31
  Maximo          : 4986.66
  Minimo          : 134.8


---
## Sección 4 — PostGIS: datos geoespaciales

PostGIS agrega a PostgreSQL la capacidad de almacenar y consultar datos geográficos.

**Flujo de esta sección:**
1. Activar la extensión PostGIS en la base de datos
2. Crear una tabla con columna de geometría
3. Insertar puntos geográficos (latitud, longitud)
4. Ejecutar consultas espaciales básicas

In [12]:
# Activar PostGIS — solo necesitas ejecutar esto una vez por base de datos
cur.execute('CREATE EXTENSION IF NOT EXISTS postgis;')
conn.commit()

# Verificar que esta activa
cur.execute("SELECT PostGIS_Version();")
v = cur.fetchone()[0]
print(f'PostGIS activo. Version: {v}')

FeatureNotSupported: extension "postgis" is not allow-listed for "azure_pg_admin" users in Azure Database for PostgreSQL
HINT:  to learn how to allow an extension or see the list of allowed extensions, please refer to https://go.microsoft.com/fwlink/?linkid=2301063


In [ ]:
# Crear tabla con columna de geometria (tipo POINT)
cur.execute("""
    CREATE TABLE IF NOT EXISTS ubicaciones (
        id       SERIAL PRIMARY KEY,
        nombre   TEXT NOT NULL,
        ciudad   TEXT,
        -- SRID 4326 = WGS84, el sistema de coordenadas de GPS y Google Maps
        geom     geometry(POINT, 4326)
    )
""")
conn.commit()
print('Tabla "ubicaciones" con columna geom (POINT, SRID 4326) creada.')

In [ ]:
# Insertar puntos geograficos con ST_MakePoint(longitud, latitud)
# Nota: el orden es longitud primero, luego latitud

puntos = [
    ('Ciudad de Mexico', 'Mexico',    -99.1332,  19.4326),
    ('Buenos Aires',     'Argentina', -58.3816, -34.6037),
    ('Bogota',           'Colombia',  -74.0721,   4.7110),
    ('Lima',             'Peru',      -77.0428, -12.0464),
    ('Santiago',         'Chile',     -70.6693, -33.4489),
    ('Madrid',           'Espana',     -3.7038,  40.4168),
]

cur.executemany("""
    INSERT INTO ubicaciones (nombre, ciudad, geom)
    VALUES (%s, %s, ST_SetSRID(ST_MakePoint(%s, %s), 4326))
""", puntos)
conn.commit()

print(f'{len(puntos)} ubicaciones insertadas.')

In [ ]:
# Consultar los puntos y exportar sus coordenadas
cur.execute("""
    SELECT
        nombre,
        ciudad,
        ST_X(geom) AS longitud,
        ST_Y(geom) AS latitud,
        ST_AsGeoJSON(geom) AS geojson
    FROM ubicaciones
    ORDER BY nombre
""")
cols = [d[0] for d in cur.description]
df_geo = pd.DataFrame(cur.fetchall(), columns=cols)
df_geo

In [ ]:
# Consulta espacial: distancia en km desde Ciudad de Mexico a cada punto
# ST_DistanceSphere calcula la distancia sobre la superficie de la Tierra

cur.execute("""
    SELECT
        nombre,
        ciudad,
        ROUND(
            ST_DistanceSphere(
                geom,
                ST_SetSRID(ST_MakePoint(-99.1332, 19.4326), 4326)
            ) / 1000
        ) AS distancia_km
    FROM ubicaciones
    WHERE nombre != 'Ciudad de Mexico'
    ORDER BY distancia_km
""")
cols = [d[0] for d in cur.description]
df_dist = pd.DataFrame(cur.fetchall(), columns=cols)
print('Distancia desde Ciudad de Mexico:')
df_dist

In [ ]:
# Consulta espacial: puntos dentro de un bounding box (caja delimitadora)
# Usamos America Latina: lat -60 a 25, lon -120 a -30

cur.execute("""
    SELECT nombre, ciudad
    FROM ubicaciones
    WHERE ST_Within(
        geom,
        ST_MakeEnvelope(-120, -60, -30, 25, 4326)
    )
    ORDER BY nombre
""")
result = cur.fetchall()
print('Puntos dentro de America Latina (bounding box):')
for row in result:
    print(f'  {row[0]} ({row[1]})')

---
## Sección 5 — Pipeline completo: Blob -> PostgreSQL

Consolidamos todo en una funcion reutilizable que:
1. Lee un CSV desde Blob Storage
2. Valida los datos con pandas
3. Inserta en PostgreSQL

In [ ]:
def pipeline_blob_a_postgres(blob_conn_str, container, archivo_csv,
                              pg_host, pg_port, pg_db, pg_user, pg_pwd):
    """
    Lee un CSV desde Blob Storage con fsspec/adlfs y lo inserta en PostgreSQL.
    Retorna el numero de filas procesadas.
    """
    import adlfs, psycopg2, pandas as pd

    # Paso 1: leer desde Blob con fsspec
    print('[1/3] Leyendo CSV desde Blob Storage...')
    parts = dict(p.split('=', 1) for p in blob_conn_str.split(';') if '=' in p)
    fs = adlfs.AzureBlobFileSystem(
        account_name=parts.get('AccountName'),
        account_key=parts.get('AccountKey')
    )
    with fs.open(f'{container}/{archivo_csv}', 'rb') as f:
        df = pd.read_csv(f)
    print(f'      {len(df)} filas leidas.')

    # Paso 2: conectar a PostgreSQL
    print('[2/3] Conectando a PostgreSQL...')
    conn = psycopg2.connect(host=pg_host, port=pg_port, dbname=pg_db,
                             user=pg_user, password=pg_pwd, sslmode='require')
    cur  = conn.cursor()

    # Paso 3: insertar
    print('[3/3] Insertando datos...')
    cur.executemany(
        'INSERT INTO registros (fecha, nombre, valor, categoria) VALUES (%s, %s, %s, %s)',
        [(str(r.iloc[0]), str(r.iloc[1]),
          float(r.iloc[2]) if pd.notna(r.iloc[2]) else None,
          str(r.iloc[3]) if len(r) > 3 else None)
         for _, r in df.iterrows()]
    )
    conn.commit()
    conn.close()
    print(f'      Pipeline completado: {len(df)} filas insertadas.')
    return len(df)


# Ejecutar
n = pipeline_blob_a_postgres(
    BLOB_CONNECTION_STRING, BLOB_CONTAINER, BLOB_ARCHIVO_CSV,
    PG_HOST, PG_PORT, PG_DATABASE, PG_USER, PG_PASSWORD
)

---
## Cierre — Cerrar conexiones

In [ ]:
try:
    cur.close()
    conn.close()
    print('Conexiones cerradas.')
except:
    print('Las conexiones ya estaban cerradas.')

---

**Resumen del notebook local:**
- Leiste un CSV desde Blob Storage sin descargarlo a disco
- Creaste e insertaste datos en una tabla PostgreSQL en Azure
- Activaste PostGIS e insertaste puntos geograficos
- Ejecutaste consultas espaciales: `ST_DistanceSphere`, `ST_Within`

Continua con el notebook de Databricks: `dia3_databricks.ipynb`